# Real-Time Garbage Detection-Classification Using YOLO 

CENG 476 – Introduction to Deep Learning, Deep Learning Project (Summer Term 2026)

This notebook consolidates all main experiments from the project into a single, reproducible pipeline. It is designed to run on **Google Colab with a GPU runtime** .

**Contents:**
1. Setup & Data Preparation
2. Baseline Model (AdamW-auto, no dropout, 60 epochs)
3. Dropout Experiment (custom architecture, p=0.2)
4. SGD Optimizer Experiment
5. Cosine LR Scheduler Experiment
6. Results Comparison & Visualization
7. Test Set Evaluation (Confusion Matrix, F1-Score)


## 1. Setup & Data Preparation

In [ ]:
# Mount Google Drive (if your dataset is stored there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics -q

### 1.1 Re-split the dataset (class-stratified 70/15/15)

**Important:** the original Roboflow split for this dataset has a severe class-imbalance
issue (e.g. the GLASS class has zero examples in the original test set). This step pools
all images/labels and creates a new, class-stratified split. See the project report,
Section 4.1 and 5.3, for details.

In [ ]:
import os, random, shutil
from collections import defaultdict

# --- CONFIGURE THESE PATHS ---
ORIGINAL_DATASET = "/content/Garbage_dataset"          
RESPLIT_DATASET  = "/content/Garbage_dataset_resplit"   
SEED = 42

def pool_dataset(input_dir, pool_img_dir, pool_lbl_dir):
    os.makedirs(pool_img_dir, exist_ok=True)
    os.makedirs(pool_lbl_dir, exist_ok=True)
    for split in ["train", "valid", "test"]:
        img_src = os.path.join(input_dir, split, "images")
        lbl_src = os.path.join(input_dir, split, "labels")
        for f in os.listdir(img_src):
            shutil.copy(os.path.join(img_src, f), os.path.join(pool_img_dir, f))
        for f in os.listdir(lbl_src):
            shutil.copy(os.path.join(lbl_src, f), os.path.join(pool_lbl_dir, f))
    print(f"Pooled {len(os.listdir(pool_img_dir))} images, {len(os.listdir(pool_lbl_dir))} labels.")

def stratified_split(pool_lbl_dir, train_ratio=0.7, val_ratio=0.15, seed=42):
    random.seed(seed)
    label_files = [f for f in os.listdir(pool_lbl_dir) if f.endswith(".txt")]
    class_to_imgs = defaultdict(list)
    for lf in label_files:
        with open(os.path.join(pool_lbl_dir, lf)) as f:
            classes = set(int(line.split()[0]) for line in f if line.strip())
        for c in classes:
            class_to_imgs[c].append(lf)

    assigned = {}
    train_set, val_set, test_set = set(), set(), set()
    for c in sorted(class_to_imgs, key=lambda c: len(class_to_imgs[c])):
        imgs = [i for i in class_to_imgs[c] if i not in assigned]
        random.shuffle(imgs)
        n = len(imgs)
        n_train, n_val = int(n * train_ratio), int(n * val_ratio)
        for i, img in enumerate(imgs):
            if i < n_train:
                train_set.add(img); assigned[img] = "train"
            elif i < n_train + n_val:
                val_set.add(img); assigned[img] = "val"
            else:
                test_set.add(img); assigned[img] = "test"
    print(f"train: {len(train_set)}, val: {len(val_set)}, test: {len(test_set)}")
    return train_set, val_set, test_set

def write_split(pool_img_dir, pool_lbl_dir, output_dir, split_name, label_files):
    img_out = os.path.join(output_dir, split_name, "images")
    lbl_out = os.path.join(output_dir, split_name, "labels")
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)
    for lf in label_files:
        img_name = lf.replace(".txt", ".jpg")
        shutil.copy(os.path.join(pool_img_dir, img_name), os.path.join(img_out, img_name))
        shutil.copy(os.path.join(pool_lbl_dir, lf), os.path.join(lbl_out, lf))

# Run the pooling + resplit
pool_img_dir = "/content/_pool/images"
pool_lbl_dir = "/content/_pool/labels"
pool_dataset(ORIGINAL_DATASET, pool_img_dir, pool_lbl_dir)
train_set, val_set, test_set = stratified_split(pool_lbl_dir, seed=SEED)
write_split(pool_img_dir, pool_lbl_dir, RESPLIT_DATASET, "train", train_set)
write_split(pool_img_dir, pool_lbl_dir, RESPLIT_DATASET, "valid", val_set)
write_split(pool_img_dir, pool_lbl_dir, RESPLIT_DATASET, "test", test_set)
print("Resplit complete:", RESPLIT_DATASET)

In [ ]:
# Write data.yaml pointing to the resplit dataset
data_yaml_content = f"""path: {RESPLIT_DATASET}
train: train/images
val: valid/images
test: test/images

nc: 6
names: ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']
"""

with open('/content/data.yaml', 'w') as f:
    f.write(data_yaml_content)

print(open('/content/data.yaml').read())

In [ ]:
# Sanity check: verify image/label counts match across splits
for split in ["train", "valid", "test"]:
    img_dir = f"{RESPLIT_DATASET}/{split}/images"
    lbl_dir = f"{RESPLIT_DATASET}/{split}/labels"
    print(split, "-> images:", len(os.listdir(img_dir)), "| labels:", len(os.listdir(lbl_dir)))

## 2. Baseline Model (AdamW-auto, no dropout, 60 epochs)

In [ ]:
from ultralytics import YOLO

model_baseline = YOLO("yolov8n.pt")

results_baseline = model_baseline.train(
    data="/content/data.yaml",
    epochs=60,
    imgsz=416,
    batch=32,
    seed=42,
    patience=15,
    project="/content/drive/MyDrive/garbage_experiments",
    name="baseline"
)

In [ ]:
# Evaluate baseline on the test set
metrics_baseline = model_baseline.val(data="/content/data.yaml", split="test")

## 3. Dropout Experiment

Custom architecture: Dropout(p=0.2) added after each of the P3/P4/P5 feature maps,
immediately before the Detect head. See `models/yolov8n_dropout.yaml` in the repo,
or the architecture diagram in the project report, Section 3.1.

In [ ]:
# Upload models/yolov8n_dropout.yaml to /content/ first, or recreate it here.
# (See the repository's models/yolov8n_dropout.yaml for the full architecture definition.)

model_dropout = YOLO("/content/yolov8n_dropout.yaml").load("yolov8n.pt")

results_dropout = model_dropout.train(
    data="/content/data.yaml",
    epochs=60,
    imgsz=416,
    batch=32,
    seed=42,
    patience=15,
    project="/content/drive/MyDrive/garbage_experiments",
    name="dropout02"
)

In [ ]:
metrics_dropout = model_dropout.val(data="/content/data.yaml", split="test")

## 4. SGD Optimizer Experiment (40 epochs)

In [ ]:
model_sgd = YOLO("yolov8n.pt")

results_sgd = model_sgd.train(
    data="/content/data.yaml",
    epochs=40,
    imgsz=416,
    batch=32,
    seed=42,
    patience=10,
    optimizer="SGD",
    lr0=0.01,
    momentum=0.937,
    project="/content/drive/MyDrive/garbage_experiments",
    name="optimizer_sgd"
)

In [ ]:
metrics_sgd = model_sgd.val(data="/content/data.yaml", split="test")

## 5. Cosine LR Scheduler Experiment (40 epochs)

In [ ]:
model_cosine = YOLO("yolov8n.pt")

results_cosine = model_cosine.train(
    data="/content/data.yaml",
    epochs=40,
    imgsz=416,
    batch=32,
    seed=42,
    patience=10,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    project="/content/drive/MyDrive/garbage_experiments",
    name="scheduler_cosine"
)

In [ ]:
metrics_cosine = model_cosine.val(data="/content/data.yaml", split="test")

## 6. Results Comparison & Visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

EXP_DIR = "/content/drive/MyDrive/garbage_experiments"

paths = {
    "Baseline": f"{EXP_DIR}/baseline/results.csv",
    "Dropout=0.2": f"{EXP_DIR}/dropout02/results.csv",
    "SGD": f"{EXP_DIR}/optimizer_sgd/results.csv",
    "Cosine Scheduler": f"{EXP_DIR}/scheduler_cosine/results.csv",
}

dfs = {}
for name, path in paths.items():
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    dfs[name] = df

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for name, df in dfs.items():
    axes[0].plot(df['epoch'], df['metrics/mAP50(B)'], label=name)
axes[0].set_title('Val mAP50 — All Experiments')
axes[0].set_xlabel('Epoch'); axes[0].legend()

for name, df in dfs.items():
    axes[1].plot(df['epoch'], df['val/box_loss'], label=name)
axes[1].set_title('Val Box Loss — All Experiments')
axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
plt.savefig(f"{EXP_DIR}/all_experiments_curves.png", dpi=150)
plt.show()

In [ ]:
# Summary table
summary = pd.DataFrame({
    "Experiment": ["Baseline", "Dropout=0.2", "SGD", "Cosine Scheduler"],
    "Epochs": [60, 60, 40, 40],
    "Test_mAP50": [metrics_baseline.box.map50, metrics_dropout.box.map50,
                   metrics_sgd.box.map50, metrics_cosine.box.map50],
    "Test_mAP50_95": [metrics_baseline.box.map, metrics_dropout.box.map,
                       metrics_sgd.box.map, metrics_cosine.box.map],
})
print(summary.to_string(index=False))
summary.to_csv(f"{EXP_DIR}/summary_table.csv", index=False)

## 7. Test Set Evaluation — Confusion Matrix & F1-Score (Baseline)

In [ ]:
# Re-run baseline validation with plots=True to generate the confusion matrix
best_model = YOLO(f"{EXP_DIR}/baseline/weights/best.pt")
final_metrics = best_model.val(data="/content/data.yaml", split="test", plots=True)

print("Confusion matrix and PR curve saved to:", final_metrics.save_dir)

In [ ]:
# Per-class F1-score
class_names = list(best_model.names.values())
p, r = final_metrics.box.p, final_metrics.box.r

print(f"{'Class':<15} {'Precision':<10} {'Recall':<10} {'F1-Score':<10}")
f1_scores = []
for i, name in enumerate(class_names):
    pi, ri = p[i], r[i]
    f1 = 2 * pi * ri / (pi + ri) if (pi + ri) > 0 else 0
    f1_scores.append(f1)
    print(f"{name:<15} {pi:<10.3f} {ri:<10.3f} {f1:<10.3f}")

print(f"\nMacro F1-Score: {sum(f1_scores)/len(f1_scores):.3f}")

## Notes

- Training was performed on Google Colab with an NVIDIA A100 GPU.
- For the real-time webcam demo (used for the project's creative component), see
  `src/webcam_demo.py` in the repository this requires a local machine with a
  physical webcam and does not run in Colab.
- See the full project report for detailed discussion of all results.
